In [1]:
!pip install -q transformers trl peft bitsandbytes accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.3 MB/s eta 0:00:00


In [2]:
from datasets import Dataset

# Structuring instruction-response pairs for climate translation
training_data = [
    {
        "instruction": "Translate this scientific IPCC finding into a public-facing policy summary and an engaging social media brief.",
        "input": "Global surface temperature was 1.09°C higher in 2011–2020 than 1850–1900. Each of the last four decades has been successively warmer than any decade that preceded it since 1850. Observed increases in well-mixed greenhouse gas concentrations are unequivocally caused by human activities.",
        "output": "### Policy Summary\nGlobal temperatures have risen by 1.09°C due to human-induced greenhouse gas emissions, with the last four decades being the warmest on record.\n\n### Social Media Brief\n🌍 The data is unequivocal: human activity is warming our planet. The last 4 decades have been hotter than any period since 1850, with global temperatures up 1.09°C. It's time for decisive climate action. #ClimateChange #ActNow"
    },
    {
        "instruction": "Translate this scientific IPCC finding into a public-facing policy summary and an engaging social media brief.",
        "input": "Climate change is already affecting many weather and climate extremes in every region across the globe. Evidence of observed changes in extremes such as heatwaves, heavy precipitation, droughts, and tropical cyclones, and, in particular, their attribution to human influence, has strengthened.",
        "output": "### Policy Summary\nHuman-driven climate change is actively intensifying extreme weather events worldwide, including heatwaves, intense rainfall, severe droughts, and tropical storms.\n\n### Social Media Brief\n⛈️ Extreme weather isn't just natural variability anymore. From severe droughts to intense heatwaves, human-caused climate change is actively amplifying extreme weather globally. Read the latest findings. #ClimateCrisis"
    }
]

# Convert to a Hugging Face Dataset object
dataset = Dataset.from_list(training_data)

# Format the text consistently into a single string for the training loop
def format_prompts(batch):
    texts = []
    for i in range(len(batch['instruction'])):
        text = f"### Instruction:\n{batch['instruction'][i]}\n\n### Input:\n{batch['input'][i]}\n\n### Response:\n{batch['output'][i]}"
        texts.append(text)
    return {"text": texts}

formatted_dataset = dataset.map(format_prompts, batched=True)
print("Instruction dataset successfully initialized and formatted!")

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Instruction dataset successfully initialized and formatted!


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-3B-Instruct"

# Configure 4-bit quantization to prevent out-of-memory errors
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
print("Base model loaded into memory with 4-bit quantization.")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base model loaded into memory with 4-bit quantization.


In [4]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=8,                     # Rank size controlling adapter complexity
    lora_alpha=16,           # Scaling factor
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], # Target attention layers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 3,686,400 || all params: 3,089,625,088 || trainable%: 0.1193


In [8]:
from trl import SFTConfig

# Define your parameter parameters inside SFTConfig
sft_config = SFTConfig(
    output_dir="./climate_model_results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    max_steps=10,
    optim="paged_adamw_8bit",
    fp16=True,
    report_to="none",
    dataset_text_field="text",
    max_length=512
)
print("Configuration options registered.")


Configuration options registered.


In [12]:
from trl import SFTConfig

# Updated configuration with fp16 disabled to prevent precision conflicts
sft_config = SFTConfig(
    output_dir="./climate_model_results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    max_steps=10,
    optim="paged_adamw_8bit",
    fp16=False,                # FIX: Set to False to let bitsandbytes handle precision
    report_to="none",
    dataset_text_field="text",
    max_length=512
)
print("Configuration options registered successfully with correct precision mappings.")

Configuration options registered successfully with correct precision mappings.


In [13]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model.base_model.model,
    train_dataset=formatted_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=sft_config,
)
print("Trainer successfully re-initialized!")

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Trainer successfully re-initialized!


In [14]:
print("Starting climate model optimization fine-tuning...")
trainer.train()
print("Fine-tuning complete! Your model has adapted to the style guidelines.")

Starting climate model optimization fine-tuning...


Step,Training Loss
1,2.194317
2,2.141439
3,2.029142
4,1.916535
5,1.829317
6,1.751789
7,1.694381
8,1.645899
9,1.614177
10,1.597535


Fine-tuning complete! Your model has adapted to the style guidelines.


In [15]:
# 1. New, unseen raw scientific text
raw_ipcc_text = "In 2019, atmospheric CO2 concentrations were higher than at any time in at least 2 million years. Concentrations of CH4 and N2O were higher than at any time in at least 800,000 years. Since 1750, increases in GHG concentrations are unequivocally caused by human activities."

# 2. Format the prompt matching the exact training layout
prompt = f"### Instruction:\nTranslate this scientific IPCC finding into a public-facing policy summary and an engaging social media brief.\n\n### Input:\n{raw_ipcc_text}\n\n### Response:\n"

# 3. Tokenize and pass to the GPU
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.3, do_sample=True)

# 4. Extract and print only the model's new response
print(tokenizer.decode(outputs[0], skip_special_tokens=True).split("### Response:\n")[-1])

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in Qwen2DecoderLayer. Setting `past_key_values=None`.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


### I a high of following the following following numbers, a number of following numbers
 a complex numbers from a                                                                                                                                                                                                                                          


Climate chat-Fine - Tuning LLM's Accessible science translation